# Baseline Model: Spectral Power for EEG Biometrics

**Objective**: Establish a performance baseline for subject identification using fundamental frequency-domain features (Band Power) and a standard Machine Learning (ML) classifier. 
This baseline quantifies the predictive power of basic physiological responses before introducing advanced time-frequency representations like Discrete Wavelet Transform (DWT).

### Step 0: Environment Setup
This section consolidates all necessary library imports, global configurations, and hyperparameter settings to ensure reproducibility and ease of debugging.

In [ ]:
# ==========================================
# Step 0: Environment Setup & Global Config
# ==========================================
import os
import glob
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.signal import welch
from tqdm.notebook import tqdm

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Global Configurations
DIR_8CH = r'D:\COGS189\EEG-BIOMETRICS-NEURAL-IDENTITY\DATA\my_preprocessed_data_8ch'
FS = 500                    # Sampling frequency (Hz)
RANDOM_SEED = 42            # For reproducibility
N_SAMPLES_PER_SUB = 2000    # Trials to subsample per subject

# Set visualization style
plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(RANDOM_SEED)

print("Environment setup complete. Libraries imported and global variables configured.")

### Step 1: Data Ingestion and Subsampling
**Behavioral Logic**:
We load the preprocessed 3D tensors `(Trials, Channels, Timepoints)` for multiple subjects. To prevent memory exhaustion (Out of Memory) and ensure a perfectly balanced dataset for the classifier, we apply uniform random subsampling. This guarantees that the baseline model learns from an equal number of single-trial samples across all classes (subjects), eliminating class imbalance bias.

In [ ]:
# ==========================================
# Step 1: Auto-Load All Subjects & Subsampling
# ==========================================
# Auto-discover all subject data files
npz_files = glob.glob(os.path.join(DIR_8CH, 'sub-*', 'preprocessed_eeg_train.npz'))
npz_files.sort()

print(f"Discovered {len(npz_files)} subject data files.")

X_list = []
y_list = []
ch_names = None

# Iterate, load, and subsample
for subject_idx, file_path in enumerate(npz_files):
    sub_name = file_path.split(os.sep)[-2] 
    
    data_dict = np.load(file_path, allow_pickle=True)
    eeg_data = data_dict['preprocessed_eeg_data'] 
    
    if subject_idx == 0:
        ch_names = data_dict['ch_names']
    
    # Random subsampling
    total_trials = eeg_data.shape[0]
    sample_indices = np.random.choice(total_trials, N_SAMPLES_PER_SUB, replace=False)
    subsampled_data = eeg_data[sample_indices]
    
    X_list.append(subsampled_data)
    y_list.extend([subject_idx] * N_SAMPLES_PER_SUB)
    
    print(f"Loaded {sub_name}: Subsampled {N_SAMPLES_PER_SUB} trials. Assigned Label: {subject_idx}")

# Concatenate into final target matrices
X_raw = np.vstack(X_list)          
y = np.array(y_list)               

print("\n==========================================")
print(f"FINAL Target Tensor X_raw shape: {X_raw.shape}")
print(f"FINAL Label Vector y shape: {y.shape}")
print("==========================================")

### Step 2: Feature Engineering (Power Spectral Density)
**Behavioral Logic**:
Raw time-series data contains excessive noise and dimensionality. Here, we transform the signal from the time domain to the frequency domain using Welch's method. We compute the Power Spectral Density (PSD) and extract the average absolute power across four standard EEG frequency bands: Theta (4-8 Hz), Alpha (8-13 Hz), Beta (13-30 Hz), and Gamma (30-40 Hz). This collapses the 3D tensor into a highly concentrated 2D feature matrix `(Samples, Features)`, capturing the fundamental harmonic resonance of the visual cortex.

In [ ]:
# ==========================================
# Step 2: Baseline Feature Extraction (Band Power)
# ==========================================
n_trials, n_channels, n_times = X_raw.shape

# Initialize 2D feature matrix: (Trials, Channels * 4 frequency bands)
X_features = np.zeros((n_trials, n_channels * 4)) 
feature_names = []

# Generate standardized feature labels
bands = ['Theta(4-8Hz)', 'Alpha(8-13Hz)', 'Beta(13-30Hz)', 'Gamma(30-40Hz)']
for ch in ch_names:
    for b in bands:
        feature_names.append(f"{ch}_{b}")

print("Extracting PSD Band Power features via Welch's Method...")

for i in tqdm(range(n_trials), desc="Processing Trials"):
    for j in range(n_channels):
        freqs, psd = welch(X_raw[i, j, :], fs=FS, nperseg=250)
        
        # Calculate mean power within specific physiological frequency bands
        theta = np.mean(psd[(freqs >= 4) & (freqs < 8)])
        alpha = np.mean(psd[(freqs >= 8) & (freqs < 13)])
        beta  = np.mean(psd[(freqs >= 13) & (freqs < 30)])
        gamma = np.mean(psd[(freqs >= 30) & (freqs < 40)])
        
        # Populate the feature matrix
        feature_idx = j * 4
        X_features[i, feature_idx:feature_idx+4] = [theta, alpha, Beta, gamma]

print(f"Feature Extraction Complete. Matrix X_features shape: {X_features.shape}")

### Step 3: Machine Learning Classifier Training
**Behavioral Logic**:
We partition the dataset into training and testing sets (80/20 split) with strict stratification to maintain class distribution. Crucially, Z-score standardization (StandardScaler) is applied to ensure that high-amplitude bands (e.g., Alpha) do not artificially dominate low-amplitude bands (e.g., Gamma) during model optimization. A Random Forest ensemble classifier is then trained to identify the nonlinear decision boundaries separating individual subjects.

In [ ]:
# ==========================================
# Step 3: Random Forest Baseline Modeling
# ==========================================
# 1. Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_features, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)

# 2. Standardization (Z-score normalization)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 3. Model Training
print("Training Random Forest Classifier (100 estimators)...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED, n_jobs=-1)
rf_model.fit(X_train_scaled, y_train)

# 4. Prediction & Metrics
y_pred = rf_model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)

unique_labels = [f"Sub-{str(i+1).zfill(2)}" for i in np.unique(y)]

print("==================================================")
print(f"BASELINE MODEL ACCURACY: {accuracy * 100:.2f}%")
print("==================================================")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=unique_labels))

# 5. Confusion Matrix Visualization
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=unique_labels, yticklabels=unique_labels)
plt.title(f'Baseline Confusion Matrix (Accuracy: {accuracy*100:.1f}%)', fontweight='bold')
plt.xlabel('Predicted Identity')
plt.ylabel('True Identity')
plt.tight_layout()
plt.show()

### Step 4: Model Interpretability and Feature Importance
**Behavioral Logic**:
A highly accurate model must be physiologically interpretable. By extracting the Gini importance metrics inherent to the Random Forest architecture, we can rank the features. This answers the core neuroscientific question: Which specific scalp locations (Channels) and neural oscillations (Frequency Bands) carry the highest concentration of subject-specific biometric fingerprints?

In [ ]:
# ==========================================
# Step 4: Feature Importance Analysis
# ==========================================
# Extract internal Gini importance weights
importances = rf_model.feature_importances_

# Construct DataFrame for mapping
feature_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

# Visualize the Top 15 driving features
top_15 = feature_df.head(15)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=top_15, palette='viridis')
plt.title('Top 15 Features Driving Biometric Identification (Baseline)', fontweight='bold')
plt.xlabel('Random Forest Feature Importance Score (Gini)')
plt.ylabel('Channel & Frequency Band')
plt.tight_layout()
plt.show()